# 02. Grouped model evaluation

Evaluates Bi/Cr/Fe by experimental series, with C01 and C02 stored separately. This notebook calls functions from `train_models.py`. Windows and channels from the same series are never split between train and test. Sample independence and shared measurement sessions still need to be checked against the laboratory records.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image

cwd = Path.cwd().resolve()
BASE_DIR = next((p for p in [cwd, *cwd.parents] if (p / "src" / "dataset_config.py").exists()), None)
if BASE_DIR is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks subdirectory")
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))
import pandas as pd
from dataset_config import FIGURES_DIR, METRICS_DIR
print(BASE_DIR)

In [ ]:
from train_models import (load_dataset, prepare_training_data, evaluate_models,
                          analyze_clusters, fit_final_model, analyze_comparison_films,
                          save_feature_plots)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
df = load_dataset()
display(df.head())

In [ ]:
df_train = prepare_training_data(df)
display(df_train.groupby(["material", "experiment_id", "channel"]).size().rename("n_windows").to_frame())

## Grouped evaluation

Scaling is fitted inside each fold. The MLP does not use an internal random split for early stopping. If a fold is missing any class, the run stops instead of reporting a misleading score.

In [ ]:
metrics, oof_predictions = evaluate_models(df_train)
display(metrics)
display(pd.read_csv(METRICS_DIR / "evaluation_splits.csv"))
for name in ("knn", "mlp"):
    display(Image(filename=str(FIGURES_DIR / f"{name}_confusion_matrix.png")))

## Exploratory clustering

Cluster numbers are not material labels. The silhouette score is computed on the training (baseline) data, and on a subsample for large datasets.

In [ ]:
silhouette = analyze_clusters(df_train)
print("Silhouette:", silhouette)
display(Image(filename=str(FIGURES_DIR / "kmeans_clusters.png")))

## Final model and comparison series

The final model is retrained on all baseline series allowed for training. This is not an additional independent estimate of accuracy. A high class probability does not show that a film is stable or that irradiation had no effect.

In [ ]:
final_model = fit_final_model(df_train)
comparison_predictions, comparison_summary = analyze_comparison_films(df, final_model)
display(comparison_summary)

In [ ]:
save_feature_plots(df)
for filename in ("file_level_gamma_comparison.png", "file_level_energy_comparison.png",
                 "gamma_boxplots_by_file.png", "energy_boxplots_by_file.png"):
    display(Image(filename=str(FIGURES_DIR / filename)))

## Results

Full predictions are in `comparison_film_predictions.csv`, and a per-file summary is in `comparison_file_summary.csv`. `irradiated_film_predictions.csv` contains only rows with confirmed `irradiated=True`; while the status is unknown, it is empty apart from the header. Running this notebook overwrites the files in `results/metrics` and the figures in `results/figures`. Older images that are no longer produced are not deleted.